# CME Futures: Portfolio Allocation

The signal stage ranks complete configurations by equal-weight validation backtest Sharpe. For
each label, this notebook retains the strongest checkpoint and signal concentration for each of
the configured number of distinct model configurations, then evaluates the declared alternative
allocators. Equal weight is not among them: it is the baseline stage itself, and because
`stage` is not part of `backtest_hash`, running it again here produces a row hashing
identically to its baseline parent, so one of the two is silently lost. Measured in this
case study's own pre-rebuild store: 48 rows stamped `stage='signal'` while carrying
`allocation.method='equal_weight'`, and no allocation-stage equal-weight rows at all.

All allocator lookbacks come from the case-study configuration. The official population is fixed
before execution; machine speed and caught failures cannot change which allocators run.

In [1]:
"""Run the declared CME futures allocation population."""

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    create_label_candidate_sets,
    open_study,
    product_universe_table,
    run_official_backtest_requests,
    shortlist_signal_configurations,
    strategy_request_frame,
)
from case_studies.utils.sweep_config import get_allocators, get_top_n_predictions

## Select signal configurations by validation Sharpe

The shortlist is deterministic. It scans the immutable signal candidate set in descending Sharpe
order with the backtest identity as tie-break, and keeps one exact checkpoint and strategy per
distinct `(family, config_name)` pair.

In [2]:
study = open_study(execution_tier="canonical")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [3]:
shortlist_size = get_top_n_predictions("cme_futures", "allocation")
allocators = get_allocators("cme_futures")
if not allocators:
    raise ValueError("the configured allocator population is empty")
if any(allocation.get("method") == "equal_weight" for allocation in allocators):
    raise ValueError(
        "equal_weight is the baseline stage, not an allocator: `stage` is not part of "
        "`backtest_hash`, so an equal-weight reweight hashes identically to its baseline "
        "parent and one of the two rows is lost. Remove it from the configured menu."
    )

request_rows = []
for label in ALL_LABELS:
    for baseline in shortlist_signal_configurations(
        study,
        label=label,
        limit=shortlist_size,
    ):
        prediction_hash = baseline.registry_record()["prediction_hash"]
        signal = baseline.spec()["strategy"]["signal"]
        for allocation in allocators:
            method = allocation["method"]
            request_rows.append(
                {
                    "request_name": f"{baseline.hash}-{method}",
                    "prediction_hash": prediction_hash,
                    "label": label,
                    "signal": signal,
                    "allocation": allocation,
                    "risk": None,
                    "costs": None,
                    "chapter": "ch17",
                }
            )
requests = strategy_request_frame(request_rows)
requests.select("request_name", "prediction_hash", "label", "signal", "allocation")

request_name,prediction_hash,label,signal,allocation
str,str,str,object,object
"""5f624ba49a05-score_weighted""","""b2fed5004b6c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}",{'method': 'score_weighted'}
"""5f624ba49a05-inverse_vol""","""b2fed5004b6c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'inverse_vol', 'vol_window': 63}"
"""5f624ba49a05-risk_parity""","""b2fed5004b6c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'risk_parity', 'vol_window': 63}"
"""5f624ba49a05-mvo_ledoit_wolf""","""b2fed5004b6c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'mvo_ledoit_wolf', 'lookback': 63}"
"""5f624ba49a05-hrp""","""b2fed5004b6c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'hrp', 'vol_window': 63}"
…,…,…,…,…
"""de1127c1a8ed-inverse_vol""","""250ba991feee""","""fwd_ret_21d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 10}","{'method': 'inverse_vol', 'vol_window': 63}"
"""de1127c1a8ed-risk_parity""","""250ba991feee""","""fwd_ret_21d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 10}","{'method': 'risk_parity', 'vol_window': 63}"
"""de1127c1a8ed-mvo_ledoit_wolf""","""250ba991feee""","""fwd_ret_21d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 10}","{'method': 'mvo_ledoit_wolf', 'lookback': 63}"


## Execute and freeze allocation candidates

Moment-based allocators receive only price history before each decision. Product-keyed typed
decisions retain the selected prediction, roll audit, expiry reference, and allocation settings.

In [4]:
execution = run_official_backtest_requests(
    study,
    requests,
    population_name="cme_futures-allocation-validation-v1",
)
candidate_sets = create_label_candidate_sets(
    study,
    execution,
    stage="allocation",
)

In [5]:
execution.catalog_rows.sort("label", "request_name")

request_name,label,prediction_hash,decision_hash,backtest_hash,complete
str,str,str,str,str,bool
"""15d4b478079c-conformal_weighte…","""fwd_ret_21d""","""bccdad952a89""","""d676a6452513""","""57826e71fbbe""",true
"""15d4b478079c-hrp""","""fwd_ret_21d""","""bccdad952a89""","""08bf0e7ca5c9""","""1736c3371d29""",true
"""15d4b478079c-inverse_vol""","""fwd_ret_21d""","""bccdad952a89""","""520534a7f742""","""9aeefc6b6d4a""",true
"""15d4b478079c-mvo_ledoit_wolf""","""fwd_ret_21d""","""bccdad952a89""","""9d8c3e97e130""","""a2367428ae9b""",true
"""15d4b478079c-risk_parity""","""fwd_ret_21d""","""bccdad952a89""","""35d72791695d""","""c46b3c851392""",true
…,…,…,…,…,…
"""c549ccb42a62-hrp""","""fwd_ret_5d""","""e5443bb55af4""","""1706d07d1e7a""","""2ec623b0574a""",true
"""c549ccb42a62-inverse_vol""","""fwd_ret_5d""","""e5443bb55af4""","""e6c843ae9cb5""","""a0d6822deedd""",true
"""c549ccb42a62-mvo_ledoit_wolf""","""fwd_ret_5d""","""e5443bb55af4""","""65cc5091f5f1""","""cabc1a68ae9b""",true


The next two execution notebooks select the highest validation Sharpe from the union of signal and
allocation results for each label. Cost sensitivity is diagnostic; risk overlays remain eligible
for final selection.